📘 Notebook – Tabela 3

Distribuição relativa das mulheres por faixa etária segundo realização de exames preventivos
PNS 2013 e 2019
Mulheres com 25 anos ou mais

🔹 Objetivo analítico

Construir a Tabela 3, apresentando a distribuição percentual de mulheres segundo faixas etárias e realização de exames preventivos:

Exame preventivo do colo do útero (Papanicolau)

Mamografia

A tabela segue rigorosamente o padrão metodológico das Tabelas 1 e 2, utilizando:

Variáveis gold já persistidas (preventivo_fez, mamografia_fez)

Percentuais relativos (%)

Total por faixa etária igual a 100%

🔹 Setup do ambiente

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path


sys.path.append(str(Path("..").resolve()))
from service.pns_service import get_dataframe


pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

🔹 Definição das faixas etárias (critério analítico)

As faixas etárias seguem o padrão clássico da literatura epidemiológica e do exemplo de referência:

In [2]:
bins = [25, 35, 45, 55, 65, np.inf]
labels = [
"Entre 25–34 anos",
"Entre 35–44 anos",
"Entre 45–54 anos",
"Entre 55–64 anos",
"Mais de 65 anos"
]

🔹 Carregamento dos dados (gold)

In [3]:
variables = [
    "idade",
    "preventivo_fez",
    "mamografia_fez",
]

sources = ["2013", "2019"]

# Filtrar mulheres (sexo=2) com idade >= 25 anos
df = get_dataframe(
    variables=variables,
    sources=sources,
    filters={"sexo": "2", "idade": {"operador": ">=", "valor": 25}}
)

🔹 Criação da variável faixa etária

In [4]:
# Criação da variável faixa etária
bins = [25, 35, 45, 55, 65, np.inf]
labels = [
    "Entre 25 e 34 anos",
    "Entre 35 e 44 anos",
    "Entre 45 e 54 anos",
    "Entre 55 e 64 anos",
    "Mais de 65 anos"
]

df["faixa_etaria"] = pd.cut(
    df["idade"],
    bins=bins,
    labels=labels,
    right=False
)

In [5]:
# Diagnóstico rápido: mostrar colunas, tipos e exemplos das variáveis usadas
print('Colunas disponíveis:', df.columns.tolist())
print('\nDtypes:')
print(df[['origem','idade','preventivo_fez','mamografia_fez']].dtypes)
print('\nAmostra:')
print(df[['origem','idade','preventivo_fez','mamografia_fez']].head(10))
# Verificar tipos das entradas nas colunas de interesse
for col in ['preventivo_fez','mamografia_fez']:
    print('\nDistinct types in', col, ':', df[col].map(type).value_counts().to_dict())
    print('Some unique values:', df[col].dropna().astype(str).unique()[:10])

Colunas disponíveis: ['origem', 'mamografia_fez', 'preventivo_fez', 'id_upa', 'id_domicilio', 'id_morador', 'idade', 'faixa_etaria']

Dtypes:
origem            object
idade              int64
preventivo_fez    object
mamografia_fez    object
dtype: object

Amostra:
  origem  idade preventivo_fez mamografia_fez
0   2013     49              1              0
1   2013     30              0              0
2   2013     29              1              0
3   2013     41              1              0
4   2013     38              0              0
5   2013     34              0              0
6   2013     37              1              0
7   2013     64              1              1
8   2013     33              0              0
9   2013     37              0              0

Distinct types in preventivo_fez : {<class 'str'>: 158912}
Some unique values: ['1' '0']

Distinct types in mamografia_fez : {<class 'str'>: 158912}
Some unique values: ['0' '1']


🔹 Função auxiliar para cálculo percentual por faixa etária

In [6]:
def tabela_percentual_por_faixa_etaria(df, col):
    # Garantir que a coluna seja numérica (algumas origens armazenam '0'/'1' como strings)
    tab = (
        df.groupby("faixa_etaria")[col]
        .agg([
            lambda x: (pd.to_numeric(x, errors="coerce").mean() * 100).round(1),
            lambda x: (100 - pd.to_numeric(x, errors="coerce").mean() * 100).round(1)
        ])
    )
    tab.columns = ["Sim", "Não"]
    tab["Total"] = 100.0
    return tab

🔹 Construção da Tabela 3 (por ano)

In [7]:
tabelas = {}

for ano in sorted(df["origem"].unique()):
    df_ano = df[df["origem"] == ano]

    tabela = pd.concat(
        {
            "Papanicolau": tabela_percentual_por_faixa_etaria(df_ano, "preventivo_fez"),
            "Mamografia": tabela_percentual_por_faixa_etaria(df_ano, "mamografia_fez"),
        },
        axis=1
    )

    tabelas[ano] = tabela

assert len(tabelas) == df["origem"].nunique(), "Nem todos os anos foram processados"



🔹 Exibição final (formato artigo)

In [8]:
for ano, tabela in tabelas.items():
    print(f"\nTabela 3 – Distribuição das mulheres por faixa etária segundo realização de exames (%), PNS {ano}")
    display(tabela)


Tabela 3 – Distribuição das mulheres por faixa etária segundo realização de exames (%), PNS 2013


Papanicolau              Mamografia             
                           Sim   Não  Total        Sim   Não  Total
faixa_etaria                                                       
Entre 25 e 34 anos        40.7  59.3  100.0        4.6  95.4  100.0
Entre 35 e 44 anos        43.0  57.0  100.0       16.7  83.3  100.0
Entre 45 e 54 anos        40.4  59.6  100.0       27.7  72.3  100.0
Entre 55 e 64 anos        42.0  58.0  100.0       31.0  69.0  100.0
Mais de 65 anos           40.3  59.7  100.0       28.8  71.2  100.0


Tabela 3 – Distribuição das mulheres por faixa etária segundo realização de exames (%), PNS 2019


Papanicolau              Mamografia             
                           Sim   Não  Total        Sim   Não  Total
faixa_etaria                                                       
Entre 25 e 34 anos        35.8  64.2  100.0        4.4  95.6  100.0
Entre 35 e 44 anos        40.3  59.7  100.0       14.9  85.1  100.0
Entre 45 e 54 anos        41.4  58.6  100.0       29.1  70.9  100.0
Entre 55 e 64 anos        45.6  54.4  100.0       34.9  65.1  100.0
Mais de 65 anos           45.5  54.5  100.0       33.0  67.0  100.0

In [11]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import textwrap

def exportar_tabela_com_descricao_pdf(tabelas, filename="Tabela_03_Artigo.pdf"):
    # Descrições específicas para cada ano
    descricoes = {
        "2013": "Análise 2013: Os dados demonstram que a realização do Papanicolaou apresentava uma cobertura homogênea acima de 40% em quase todas as faixas, exceto na mais idosa. Já a mamografia mostrava um forte gradiente etário, concentrando-se a partir dos 45 anos, evidenciando o início das políticas de rastreamento da época.",
        "2019": "Análise 2019: Observa-se um aumento na cobertura de mamografia na faixa prioritária (55-64 anos), atingindo 34,9%. O exame de Papanicolaou manteve-se estável, com leve crescimento entre mulheres acima de 55 anos, sugerindo uma melhoria na continuidade do cuidado preventivo ao longo do envelhecimento feminino."
    }

    with PdfPages(filename) as pdf:
        for ano, df_tabela in tabelas.items():
            # Aumentamos o tamanho da figura para caber o texto (largura, altura)
            fig, ax = plt.subplots(figsize=(12, 8))
            ax.axis('tight')
            ax.axis('off')
            
            # 1. Título
            plt.text(0.5, 0.95, f"Tabela 3 – Distribuição das mulheres por faixa etária segundo realização de exames (%), PNS {ano}", 
                     horizontalalignment='center', fontsize=12, fontweight='bold', transform=ax.transAxes)

            # 2. Preparação da Tabela
            df_plot = df_tabela.reset_index()
            col_labels = ['Faixa Etária', 'Sim (Pap)', 'Não (Pap)', 'Total', 'Sim (Mam)', 'Não (Mam)', 'Total']
            
            the_table = ax.table(cellText=df_plot.values, 
                                 colLabels=col_labels, 
                                 loc='center', 
                                 cellLoc='center')
            
            # Estilização da Tabela
            the_table.auto_set_font_size(False)
            the_table.set_fontsize(10)
            the_table.scale(1.1, 2) # Ajusta escala
            
            # Cabeçalho em negrito e cor de fundo leve
            for (row, col), cell in the_table.get_celld().items():
                if row == 0:
                    cell.set_text_props(weight='bold')
                    cell.set_facecolor('#f2f2f2')

            # 3. Descrição Analítica (abaixo da tabela)
            texto_analise = descricoes.get(ano, "")
            # Quebra o texto em linhas de no máximo 100 caracteres
            linhas_texto = textwrap.fill(texto_analise, width=100)
            
            plt.text(0.05, 0.20, "Descrição Analítica:", fontweight='bold', fontsize=10, transform=ax.transAxes)
            plt.text(0.05, 0.08, linhas_texto, fontsize=10, verticalalignment='top', transform=ax.transAxes, style='italic')

            # 4. Fonte
            plt.figtext(0.1, 0.05, "Fonte: Elaboração própria a partir dos microdados da PNS (IBGE).", fontsize=8)
            
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
            
    print(f"Sucesso! O arquivo '{filename}' foi gerado com as descrições.")

# Executar a função
exportar_tabela_com_descricao_pdf(tabelas)

Sucesso! O arquivo 'Tabela_03_Artigo.pdf' foi gerado com as descrições.


📌 Nota metodológica

O denominador de cada faixa etária corresponde ao total de mulheres naquela faixa;

As categorias “Sim” e “Não” referem-se exclusivamente à realização do exame, conforme definição das variáveis gold;

Os percentuais são calculados intra-faixa etária, totalizando 100% em cada linha;

Não são realizadas imputações nem ponderações adicionais além do critério já consolidado nas Tabelas 1 e 2.

Fonte: Elaboração própria a partir dos microdados da Pesquisa Nacional de Saúde (PNS 2013 e 2019), IBGE.